In [ ]:
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
# ==========================================
# Milestone 1: Setup, Installation, and Model Loading
# ==========================================

# Step 1: Check GPU availability
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Step 2: Install Required Libraries
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth==2025.9.2
!pip install transformers==4.55.4
!pip install --no-deps trl==0.22.2

# Step 3: Import Libraries and Set Random Seed
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template
torch.manual_seed(42)  # Set for reproducibility

# Step 4: Define Model Parameters
max_seq_length = 256
dtype = None
load_in_4bit = True

# Step 5: Load the Pre-trained Llama 3.2 Model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-1B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# Step 6: Apply LoRA Adapters to the Model
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)
print(f"Loaded unsloth/Llama-3.2-1B-Instruct with LoRA adapters on {device}")

Using device: cuda
  Using cached xformers-0.0.29.post3.tar.gz (8.5 MB)
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
ERROR: Failed to build 'xformers' when getting requirements to build wheel
==((====))==  Unsloth 2025.9.2: Fast Llama patching. Transformers: 4.55.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Loaded unsloth/Llama-3.2-1B-Instruct with LoRA adapters on cuda


In [ ]:
# ==========================================
# Milestone 2: Data Preparation & Formatting
# ==========================================

import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

# Step 1: Load the Dataset into a Pandas dataframe
df = pd.read_csv("Lead Scoring.csv")

# Step 2: Choose relevant columns and clean data
text_columns = [
    'Lead Origin', 'Lead Source', 'Last Activity', 'Country', 'Specialization',
    'How did you hear about X Education', 'What is your current occupation',
    'What matters most to you in choosing a course', 'Tags'
]
df = df[text_columns + ['Converted']].dropna()
print(f"Dataset size after dropping NaNs: {len(df)} rows")

# Step 3: Split the Data into Train and Validation Sets
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['Converted'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['Converted'])
print(f"Train size: {len(train_df)}, Validation size: {len(val_df)}")

# Step 4: Create Conversational Prompts for Each Lead
def create_conversations(dataframe):
    conversations = []
    for _, row in dataframe.iterrows():
        lead_info = "".join([f"{col}: {row[col]}\n" for col in text_columns])
        convo = [
            {"role": "user", "content": f"Analyze this lead information and classify if they will convert:\n\n{lead_info}"},
            {"role": "assistant", "content": "1" if row['Converted'] == 1 else "0"}
        ]
        conversations.append({"conversations": convo})
    return conversations

train_dataset = Dataset.from_list(create_conversations(train_df))
val_dataset = Dataset.from_list(create_conversations(val_df))

# Step 5: Apply Chat Template to format text (Without manual pre-tokenization)
tokenizer = get_chat_template(tokenizer, chat_template="llama-3.2")

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
val_dataset = val_dataset.map(formatting_prompts_func, batched=True)

print("Milestone 2 completed successfully: Data prepared and formatted!")

Dataset size after dropping NaNs: 4534 rows
Train size: 3627, Validation size: 453


Map:   0%|          | 0/3627 [00:00<?, ? examples/s]

Map:   0%|          | 0/453 [00:00<?, ? examples/s]

Milestone 2 completed successfully: Data prepared and formatted!


In [ ]:
!pip install --upgrade torchvision

In [ ]:
# ==========================================
# Milestone 3: Model Training (SFTTrainer)
# ==========================================

from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# Step 1: Configure Training Arguments and SFTTrainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False, # Can speed up training for short sequences
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        eval_strategy="steps",
        eval_steps=20,
        report_to="none",
    ),
)

# Step 2: Train the Model
trainer_stats = trainer.train()
print("Training completed successfully!")

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/3627 [00:00<?, ? examples/s]

Exception ignored in: <_io.BytesIO object at 0x79272c7d5e90>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/tblib/__init__.py", line 139, in as_traceback
    next_tb = sys.exc_info()[2].tb_next
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x79272c7d6e80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/tblib/__init__.py", line 139, in as_traceback
    next_tb = sys.exc_info()[2].tb_next
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x79272c7cbbf0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/tblib/__init__.py", line 139, in as_traceback
    next_tb = sys.exc_info()[2].tb_next
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x79272c7d6ed0>
Traceback (most recent call last):
  File "/usr/local/

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/453 [00:00<?, ? examples/s]

Exception ignored in: <_io.BytesIO object at 0x79272c7cbce0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/filelock/_api.py", line 2221, in _audit_fork_safety
    def _audit_fork_safety(  # pragma: no cover - CPython disables tracing while Python audit hooks run
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x79272c771940>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/filelock/_api.py", line 2221, in _audit_fork_safety
    def _audit_fork_safety(  # pragma: no cover - CPython disables tracing while Python audit hooks run
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x79272c76f470>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/filelock/_api.py", line 2221, in _audit_fork_safety
    def _audit_fork_safety(  # pragma: no cover - CPython disables tracing

Step,Training Loss,Validation Loss
20,0.355400,0.405772
40,0.358100,0.272182
60,0.245400,0.236617


Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Training completed successfully!


In [ ]:
# ==========================================
# Milestone 4: Inference & Evaluation (Final & Clean)
# ==========================================

import torch
from sklearn.metrics import accuracy_score, confusion_matrix
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

# ------------------------------------------
# Step 1: Select a Test Record for Prediction
# ------------------------------------------
test_record = test_df.iloc[0].to_dict()
test_lead = {k: v for k, v in test_record.items() if k != 'Converted'}
print("Sample Test Lead Selected:")
print(test_record)

# ------------------------------------------
# Step 2: Load the Pre-trained Model for Few-Shot Baseline
# ------------------------------------------
def load_pretrained_model(max_seq_length=8000):
    print("\nLoading pre-trained model for few-shot baseline...")
    pretrained_model, pretrained_tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/Llama-3.2-1B-Instruct",
        max_seq_length=max_seq_length,
        dtype=None,
        load_in_4bit=True,
    )
    pretrained_tokenizer = get_chat_template(pretrained_tokenizer, chat_template="llama-3.2")
    FastLanguageModel.for_inference(pretrained_model)
    return pretrained_model, pretrained_tokenizer

pretrained_model, pretrained_tokenizer = load_pretrained_model()

# ------------------------------------------
# Step 3: Predict with Few-Shot Model
# ------------------------------------------
def predict_with_pretrained(model, tokenizer, lead_data, device=device, examples=""):
    system_prompt = f"""You are a lead conversion classifier. Predict whether the given lead will convert, given the available lead information. Respond with '1' for yes (will convert) or '0' for no (will not convert).

    **Examples:**

    {str(examples)}

    IMPORTANT:
    - Skip pre- and post text.
    - Do NOT provide explanations
    - Just output the prediction (0 or 1)
    """

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Lead:\n{lead_data}\nClassification:"}
    ]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False)

    inputs = tokenizer(prompt_text, padding=True, truncation=True, max_length=7000, return_tensors="pt").to(device)
    input_length = inputs.input_ids.shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            pad_token_id=tokenizer.eos_token_id,
            max_new_tokens=50,
            temperature=0.0,
            do_sample=False,
            return_dict_in_generate=True
        )
        new_tokens = outputs.sequences[0, input_length:]
        response_text = tokenizer.decode(new_tokens, skip_special_tokens=True)

    response_text = response_text.strip("assistant\n\n")

    if response_text != "1" and response_text != "0":
        return "0"
    return response_text

examples = train_df.to_dict(orient='records')[:10]
pretrained_result = predict_with_pretrained(pretrained_model, pretrained_tokenizer, lead_data=test_lead, examples=examples)
print(f"Few-shot prediction for sample: {pretrained_result}")

# ------------------------------------------
# Step 4: Prepare Fine-Tuned Model (Using active trained 'model' from memory)
# ------------------------------------------
print("\nPreparing fine-tuned model for inference...")
FastLanguageModel.for_inference(model)
finetuned_model = model
finetuned_tokenizer = tokenizer

# ------------------------------------------
# Step 5: Predict with Fine-Tuned Model
# ------------------------------------------
def predict_with_finetuned(model, tokenizer, lead_data, device=device):
    prompt = [{"role": "user", "content": f"Analyze this lead information and classify if they will convert (answer with '1' for yes, '0' for no):\n\n{lead_data}"}]
    prompt_text = tokenizer.apply_chat_template(prompt, tokenize=False)

    inputs = tokenizer(prompt_text, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
    input_length = inputs.input_ids.shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            pad_token_id=tokenizer.eos_token_id,
            max_new_tokens=10,
            temperature=0.0,
            do_sample=False,
            return_dict_in_generate=True
        )
        new_tokens = outputs.sequences[0, input_length:]
        response_text = tokenizer.decode(new_tokens, skip_special_tokens=True)

    response_text = response_text.strip("assistant\n\n")

    if response_text != "1" and response_text != "0":
        return "0"
    return response_text

finetuned_result = predict_with_finetuned(finetuned_model, finetuned_tokenizer, test_lead, device)
print(f"Fine-tuned prediction for sample: {finetuned_result}")

# ------------------------------------------
# Step 6: Evaluate Model Performance
# ------------------------------------------
def evaluate_model(predict_fn, model, tokenizer, test_df, model_name="Model", sample_size=100, device=device):
    if sample_size and len(test_df) > sample_size:
        eval_df = test_df.sample(sample_size, random_state=42)
        print(f"\nEvaluating {model_name} on {sample_size} random samples...")
    else:
        eval_df = test_df
        print(f"\nEvaluating {model_name} on all {len(test_df)} samples...")

    y_true = []
    y_pred = []

    for idx, record in enumerate(eval_df.to_dict(orient='records')):
        actual = int(record['Converted'])
        y_true.append(actual)
        lead_data = "\n".join([f"{k}: {v}" for k, v in record.items() if k != 'Converted'])
        prediction = predict_fn(model, tokenizer, lead_data, device)
        pred_binary = 1 if prediction == "1" else 0
        y_pred.append(pred_binary)
        if (idx + 1) % 25 == 0:
            print(f"Processed {idx + 1}/{len(eval_df)} records...")

    accuracy = accuracy_score(y_true, y_pred)
    print(f"{model_name} Accuracy: {accuracy:.4f}")

    return {
        'model_name': model_name,
        'accuracy': accuracy,
        'confusion_matrix': confusion_matrix(y_true, y_pred)
    }

few_shot_results = evaluate_model(predict_with_pretrained,
                                  pretrained_model,
                                  pretrained_tokenizer,
                                  test_df,
                                  "Few-Shot Model",
                                  sample_size=100)

finetuned_results = evaluate_model(predict_with_finetuned,
                                   finetuned_model,
                                   finetuned_tokenizer,
                                   test_df,
                                   "Fine-Tuned Model",
                                   sample_size=100)

# ------------------------------------------
# Step 7: Compare Results
# ------------------------------------------
print("\n================================")
print("Model Comparison Results:")
print("================================")
print(f"Few-Shot Accuracy: {few_shot_results['accuracy']:.4f}")
print(f"Fine-Tuned Accuracy: {finetuned_results['accuracy']:.4f}")
print(f"Improvement: {finetuned_results['accuracy'] - few_shot_results['accuracy']:.4f}")
print("================================")

Sample Test Lead Selected:
{'Lead Origin': 'Landing Page Submission', 'Lead Source': 'Direct Traffic', 'Last Activity': 'Email Opened', 'Country': 'India', 'Specialization': 'Banking, Investment And Insurance', 'How did you hear about X Education': 'Select', 'What is your current occupation': 'Unemployed', 'What matters most to you in choosing a course': 'Better Career Prospects', 'Tags': 'Ringing', 'Converted': 0}

Loading pre-trained model for few-shot baseline...
==((====))==  Unsloth 2025.9.2: Fast Llama patching. Transformers: 4.55.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Few-shot prediction for sample: 1

Preparing fine-tuned model for inference...
F

In [ ]:
# ==========================================
# Milestone 2: Data Preparation & Formatting (Corrected & Aligned)
# ==========================================

import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

# Step 1: Load the Dataset into a Pandas dataframe
df = pd.read_csv("Lead Scoring.csv")

# Step 2: Choose relevant columns and clean data
text_columns = [
    'Lead Origin', 'Lead Source', 'Last Activity', 'Country', 'Specialization',
    'How did you hear about X Education', 'What is your current occupation',
    'What matters most to you in choosing a course', 'Tags'
]
df = df[text_columns + ['Converted']].dropna()
print(f"Dataset size after dropping NaNs: {len(df)} rows")

# Step 3: Split the Data into Train, Validation, and Test Sets
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['Converted'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['Converted'])
print(f"Train size: {len(train_df)}, Validation size: {len(val_df)}, Test size: {len(test_df)}")

# Step 4: Create Conversational Prompts for Each Lead (مع إضافة الـ label هنا)
def create_conversations(dataframe):
    conversations = []
    for _, row in dataframe.iterrows():
        lead_info = "".join([f"{col}: {row[col]}\n" for col in text_columns])
        is_converted = row['Converted']
        convo = [
            {"role": "user", "content": f"Analyze this lead information and classify if they will convert:\n\n{lead_info}"},
            {"role": "assistant", "content": "1" if is_converted == 1 else "0"}
        ]
        # التعديل الهام: إضافة label لكي تراه الخطوات القادمة
        conversations.append({"conversations": convo, "label": is_converted})
    return conversations

train_dataset = Dataset.from_list(create_conversations(train_df))
val_dataset = Dataset.from_list(create_conversations(val_df))
test_dataset = Dataset.from_list(create_conversations(test_df))

# Step 5: Apply Chat Template to format text and map labels
tokenizer = get_chat_template(tokenizer, chat_template="llama-3.2")

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    # التعديل الهام: إرجاع text و labels معاً
    return {"text": texts, "labels": examples["label"]}

train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
val_dataset = val_dataset.map(formatting_prompts_func, batched=True)
test_dataset = test_dataset.map(formatting_prompts_func, batched=True)

# Step 6: Tokenize data and set PyTorch format
def tokenize_data(examples):
    return tokenizer(examples["text"], padding=True, truncation=True, max_length=256)

train_dataset = train_dataset.map(tokenize_data, batched=True)
val_dataset = val_dataset.map(tokenize_data, batched=True)
test_dataset = test_dataset.map(tokenize_data, batched=True)

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
val_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

print("Milestone 2 completed successfully: Data prepared, formatted, and tokenized!")

Dataset size after dropping NaNs: 4534 rows
Train size: 3627, Validation size: 453, Test size: 454


Map:   0%|          | 0/3627 [00:00<?, ? examples/s]

Map:   0%|          | 0/453 [00:00<?, ? examples/s]

Map:   0%|          | 0/454 [00:00<?, ? examples/s]

Map:   0%|          | 0/3627 [00:00<?, ? examples/s]

Map:   0%|          | 0/453 [00:00<?, ? examples/s]

Map:   0%|          | 0/454 [00:00<?, ? examples/s]

Milestone 2 completed successfully: Data prepared, formatted, and tokenized!


In [ ]:
!pip install --upgrade torchvision

In [ ]:
# ==========================================
# Milestone 2: Data Preparation & Formatting (Corrected & Aligned)
# ==========================================

import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

# Step 1: Load the Dataset into a Pandas dataframe
df = pd.read_csv("Lead Scoring.csv")

# Step 2: Choose relevant columns and clean data
text_columns = [
    'Lead Origin', 'Lead Source', 'Last Activity', 'Country', 'Specialization',
    'How did you hear about X Education', 'What is your current occupation',
    'What matters most to you in choosing a course', 'Tags'
]
df = df[text_columns + ['Converted']].dropna()
print(f"Dataset size after dropping NaNs: {len(df)} rows")

# Step 3: Split the Data into Train, Validation, and Test Sets
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['Converted'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['Converted'])
print(f"Train size: {len(train_df)}, Validation size: {len(val_df)}, Test size: {len(test_df)}")

# Step 4: Create Conversational Prompts for Each Lead (مع إضافة الـ label هنا)
def create_conversations(dataframe):
    conversations = []
    for _, row in dataframe.iterrows():
        lead_info = "".join([f"{col}: {row[col]}\n" for col in text_columns])
        is_converted = row['Converted']
        convo = [
            {"role": "user", "content": f"Analyze this lead information and classify if they will convert:\n\n{lead_info}"},
            {"role": "assistant", "content": "1" if is_converted == 1 else "0"}
        ]
        # التعديل الهام: إضافة label لكي تراه الخطوات القادمة
        conversations.append({"conversations": convo, "label": is_converted})
    return conversations

train_dataset = Dataset.from_list(create_conversations(train_df))
val_dataset = Dataset.from_list(create_conversations(val_df))
test_dataset = Dataset.from_list(create_conversations(test_df))

# Step 5: Apply Chat Template to format text and map labels
tokenizer = get_chat_template(tokenizer, chat_template="llama-3.2")

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    # التعديل الهام: إرجاع text و labels معاً
    return {"text": texts, "labels": examples["label"]}

train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
val_dataset = val_dataset.map(formatting_prompts_func, batched=True)
test_dataset = test_dataset.map(formatting_prompts_func, batched=True)

# Step 6: Tokenize data and set PyTorch format
def tokenize_data(examples):
    return tokenizer(examples["text"], padding=True, truncation=True, max_length=256)

train_dataset = train_dataset.map(tokenize_data, batched=True)
val_dataset = val_dataset.map(tokenize_data, batched=True)
test_dataset = test_dataset.map(tokenize_data, batched=True)

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
val_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

print("Milestone 2 completed successfully: Data prepared, formatted, and tokenized!")

Dataset size after dropping NaNs: 4534 rows
Train size: 3627, Validation size: 453, Test size: 454


Map:   0%|          | 0/3627 [00:00<?, ? examples/s]

Map:   0%|          | 0/453 [00:00<?, ? examples/s]

Map:   0%|          | 0/454 [00:00<?, ? examples/s]

Map:   0%|          | 0/3627 [00:00<?, ? examples/s]

Map:   0%|          | 0/453 [00:00<?, ? examples/s]

Map:   0%|          | 0/454 [00:00<?, ? examples/s]

Milestone 2 completed successfully: Data prepared, formatted, and tokenized!


In [ ]:
# ==========================================
# Milestone 2: Data Preparation & Formatting (Corrected & Aligned)
# ==========================================

import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

# Step 1: Load the Dataset into a Pandas dataframe
df = pd.read_csv("Lead Scoring.csv")

# Step 2: Choose relevant columns and clean data
text_columns = [
    'Lead Origin', 'Lead Source', 'Last Activity', 'Country', 'Specialization',
    'How did you hear about X Education', 'What is your current occupation',
    'What matters most to you in choosing a course', 'Tags'
]
df = df[text_columns + ['Converted']].dropna()
print(f"Dataset size after dropping NaNs: {len(df)} rows")

# Step 3: Split the Data into Train, Validation, and Test Sets
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['Converted'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['Converted'])
print(f"Train size: {len(train_df)}, Validation size: {len(val_df)}, Test size: {len(test_df)}")

# Step 4: Create Conversational Prompts for Each Lead (مع إضافة الـ label هنا)
def create_conversations(dataframe):
    conversations = []
    for _, row in dataframe.iterrows():
        lead_info = "".join([f"{col}: {row[col]}\n" for col in text_columns])
        is_converted = row['Converted']
        convo = [
            {"role": "user", "content": f"Analyze this lead information and classify if they will convert:\n\n{lead_info}"},
            {"role": "assistant", "content": "1" if is_converted == 1 else "0"}
        ]
        # التعديل الهام: إضافة label لكي تراه الخطوات القادمة
        conversations.append({"conversations": convo, "label": is_converted})
    return conversations

train_dataset = Dataset.from_list(create_conversations(train_df))
val_dataset = Dataset.from_list(create_conversations(val_df))
test_dataset = Dataset.from_list(create_conversations(test_df))

# Step 5: Apply Chat Template to format text and map labels
tokenizer = get_chat_template(tokenizer, chat_template="llama-3.2")

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    # التعديل الهام: إرجاع text و labels معاً
    return {"text": texts, "labels": examples["label"]}

train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
val_dataset = val_dataset.map(formatting_prompts_func, batched=True)
test_dataset = test_dataset.map(formatting_prompts_func, batched=True)

# Step 6: Tokenize data and set PyTorch format
def tokenize_data(examples):
    return tokenizer(examples["text"], padding=True, truncation=True, max_length=256)

train_dataset = train_dataset.map(tokenize_data, batched=True)
val_dataset = val_dataset.map(tokenize_data, batched=True)
test_dataset = test_dataset.map(tokenize_data, batched=True)

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
val_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

print("Milestone 2 completed successfully: Data prepared, formatted, and tokenized!")

Dataset size after dropping NaNs: 4534 rows
Train size: 3627, Validation size: 453, Test size: 454


Map:   0%|          | 0/3627 [00:00<?, ? examples/s]

Map:   0%|          | 0/453 [00:00<?, ? examples/s]

Map:   0%|          | 0/454 [00:00<?, ? examples/s]

Map:   0%|          | 0/3627 [00:00<?, ? examples/s]

Map:   0%|          | 0/453 [00:00<?, ? examples/s]

Map:   0%|          | 0/454 [00:00<?, ? examples/s]

Milestone 2 completed successfully: Data prepared, formatted, and tokenized!


In [ ]:
# ==========================================
# Milestone 3: Fine-Tuning the Model (Fixed)
# ==========================================

import datasets
# تعطيل فحص تورش فيجن لتجنب خطأ VideoReader نهائياً
datasets.config.TORCHVISION_AVAILABLE = False

# Step 1: Import Additional Libraries for Training
from trl import SFTTrainer, SFTConfig

# Step 2: Simplify Dataset Columns
train_dataset = train_dataset.select_columns(["text", "labels", "input_ids", "attention_mask"])
val_dataset = val_dataset.select_columns(["text", "labels", "input_ids", "attention_mask"])
test_dataset = test_dataset.select_columns(["text", "labels", "input_ids", "attention_mask"])

print("After simplification - Train dataset keys:", train_dataset.column_names)

# Step 3: Configure the SFTTrainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset = train_dataset.remove_columns(["labels"]),
    max_seq_length=256,
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=1,
        max_steps=100,
        learning_rate=2e-4,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

# Step 4: Train the Model
print("Starting training...")
trainer_stats = trainer.train()
print(f"Training completed. Steps: {trainer_stats.global_step}, Loss (avg): {trainer_stats.training_loss:.4f}")

# Step 5: Prepare Model for Inference and Save
FastLanguageModel.for_inference(model)
model.save_pretrained("lead_scoring_model")
tokenizer.save_pretrained("lead_scoring_model")
print("Model saved to ./lead_scoring_model successfully!")

After simplification - Train dataset keys: ['text', 'labels', 'input_ids', 'attention_mask']
Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,627 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Step,Training Loss
10,4.425900
20,1.455700
30,0.840200
40,0.759200
50,0.694000
60,0.667800
70,0.586300
80,0.570500
90,0.510500
100,0.504700


Training completed. Steps: 100, Loss (avg): 1.1015
Model saved to ./lead_scoring_model successfully!


In [ ]:
# ==========================================
# Milestone 4: Inference and Evaluation
# ==========================================

import torch
from sklearn.metrics import accuracy_score, confusion_matrix
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

# Step 1: Select a Test Record for Prediction
test_record = test_df.iloc[0].to_dict()
test_lead = {k: v for k, v in test_record.items() if k != 'Converted'}
print("Sample Test Lead Record:")
print(test_record)

# Step 2: Load the Pre-trained Model for Few-Shot Baseline
def load_pretrained_model(max_seq_length=8000):
    print("\nLoading pre-trained model for few-shot baseline...")
    pretrained_model, pretrained_tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/Llama-3.2-1B-Instruct",
        max_seq_length=max_seq_length,
        dtype=None,
        load_in_4bit=True,
    )
    pretrained_tokenizer = get_chat_template(pretrained_tokenizer, chat_template="llama-3.2")
    FastLanguageModel.for_inference(pretrained_model)
    return pretrained_model, pretrained_tokenizer

pretrained_model, pretrained_tokenizer = load_pretrained_model()

# Step 3: Predict with Few-Shot Model
def predict_with_pretrained(model, tokenizer, lead_data, device=device, examples=""):
    system_prompt = f"""You are a lead conversion classifier. Predict whether the given lead will convert, given the available lead information. Respond with '1' for yes (will convert) or '0' for no (will not convert).

    **Examples:**

    {str(examples)}

    IMPORTANT:
    - Skip pre- and post text.
    - Do NOT provide explanations
    - Just output the prediction (0 or 1)
    """

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Lead:\n{lead_data}\nClassification:"}
    ]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False)

    inputs = tokenizer(prompt_text, padding=True, truncation=True, max_length=7000, return_tensors="pt").to(device)
    input_length = inputs.input_ids.shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            pad_token_id=tokenizer.eos_token_id,
            max_new_tokens=50,
            temperature=0.0,
            do_sample=False,
            return_dict_in_generate=True
        )
        new_tokens = outputs.sequences[0, input_length:]
        response_text = tokenizer.decode(new_tokens, skip_special_tokens=True)

    response_text = response_text.strip("assistant\n\n")

    if response_text != "1" and response_text != "0":
        return "0"
    return response_text

examples = train_df.to_dict(orient='records')[:10]
pretrained_result = predict_with_pretrained(pretrained_model, pretrained_tokenizer, lead_data=test_lead, examples=examples)
print(f"Few-shot prediction: {pretrained_result}")

import os

# Step 4: Load the Fine-Tuned Model Safely from Local Directory
from peft import PeftModel

def load_finetuned_model(max_seq_length=256, device=device):
    print("Loading fine-tuned model and adapters from local directory...")

    # 1. تحميل الموديل الأساسي الأصلي بنفس المواصفات
    base_model, finetuned_tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/Llama-3.2-1B-Instruct",
        max_seq_length=max_seq_length,
        dtype=None,
        load_in_4bit=True,
    )

    # 2. ربط الـ Adapters المحفوظة محلياً في مجلد lead_scoring_model
    finetuned_model = PeftModel.from_pretrained(base_model, "lead_scoring_model")

    finetuned_tokenizer = get_chat_template(finetuned_tokenizer, chat_template="llama-3.2")
    FastLanguageModel.for_inference(finetuned_model)
    return finetuned_model, finetuned_tokenizer

finetuned_model, finetuned_tokenizer = load_finetuned_model()

# Step 5: Predict with Fine-Tuned Model
def predict_with_finetuned(model, tokenizer, lead_data, device=device):
    prompt = [{"role": "user", "content": f"Analyze this lead information and classify if they will convert (answer with '1' for yes, '0' for no):\n\n{lead_data}"}]
    prompt_text = tokenizer.apply_chat_template(prompt, tokenize=False)

    inputs = tokenizer(prompt_text, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
    input_length = inputs.input_ids.shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            pad_token_id=tokenizer.eos_token_id,
            max_new_tokens=10,
            temperature=0.0,
            do_sample=False,
            return_dict_in_generate=True
        )
        new_tokens = outputs.sequences[0, input_length:]
        response_text = tokenizer.decode(new_tokens, skip_special_tokens=True)

    response_text = response_text.strip("assistant\n\n")

    if response_text != "1" and response_text != "0":
        return "0"
    return response_text

finetuned_result = predict_with_finetuned(finetuned_model, finetuned_tokenizer, test_lead, device)
print(f"Fine-tuned prediction: {finetuned_result}")

# Step 6: Evaluate and Compare Model Performance
def evaluate_model(predict_fn, model, tokenizer, test_df, model_name="Model", sample_size=100, device=device):
    if sample_size and len(test_df) > sample_size:
        eval_df = test_df.sample(sample_size, random_state=42)
        print(f"\nEvaluating {model_name} on {sample_size} random samples...")
    else:
        eval_df = test_df
        print(f"\nEvaluating {model_name} on all {len(test_df)} samples...")

    y_true = []
    y_pred = []

    for idx, record in enumerate(eval_df.to_dict(orient='records')):
        actual = int(record['Converted'])
        y_true.append(actual)
        lead_data = "\n".join([f"{k}: {v}" for k, v in record.items() if k != 'Converted'])
        prediction = predict_fn(model, tokenizer, lead_data, device)
        pred_binary = 1 if prediction == "1" else 0
        y_pred.append(pred_binary)
        if (idx + 1) % 25 == 0:
            print(f"Processed {idx + 1}/{len(eval_df)} records...")

    accuracy = accuracy_score(y_true, y_pred)
    print(f"{model_name} Accuracy: {accuracy:.4f}")

    return {
        'model_name': model_name,
        'accuracy': accuracy,
        'confusion_matrix': confusion_matrix(y_true, y_pred)
    }

few_shot_results = evaluate_model(predict_with_pretrained, pretrained_model, pretrained_tokenizer, test_df, "Few-Shot Model", sample_size=100)
finetuned_results = evaluate_model(predict_with_finetuned, finetuned_model, finetuned_tokenizer, test_df, "Fine-Tuned Model", sample_size=100)

# Step 7: Final Comparison Output
print("\n" + "="*30)
print("Final Model Comparison Results:")
print("="*30)
print(f"Few-Shot Accuracy: {few_shot_results['accuracy']:.4f}")
print(f"Fine-Tuned Accuracy: {finetuned_results['accuracy']:.4f}")
print(f"Improvement / Performance Gain: {finetuned_results['accuracy'] - few_shot_results['accuracy']:.4f}")
print("="*30)

Sample Test Lead Record:
{'Lead Origin': 'Landing Page Submission', 'Lead Source': 'Direct Traffic', 'Last Activity': 'Email Opened', 'Country': 'India', 'Specialization': 'Banking, Investment And Insurance', 'How did you hear about X Education': 'Select', 'What is your current occupation': 'Unemployed', 'What matters most to you in choosing a course': 'Better Career Prospects', 'Tags': 'Ringing', 'Converted': 0}

Loading pre-trained model for few-shot baseline...
==((====))==  Unsloth 2025.9.2: Fast Llama patching. Transformers: 4.55.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Few-shot prediction: 1
Loading fine-tuned model and adapters from local directory.